In [ ]:
#Prueba de carga de los 2 ficheros (.parquet + .csv) con pandas
import pandas as pd
import time
import os

#0. Path de los ficheros
parquet_file = r"C:\PCAD\yellow_tripdata_2025-01.parquet"
csv_file = r"C:\PCAD\taxis.csv"

#1. Medir el tiempo de carga del fichero parquet
start_parquet = time.time()
df_parquet = pd.read_parquet(parquet_file)
end_parquet = time.time()
tiempo_parquet = end_parquet - start_parquet

#2. Medir el tiempo de cargar del fichero csv
start_csv = time.time()
df_csv = pd.read_csv(csv_file)
end_csv = time.time()
tiempo_csv = end_csv - start_csv

#3. Mostrar los resultados
print(f"Tiempo de carga con parquet | pandas: {tiempo_parquet:.4f} segundos")
print(f"Tiempo de carga con csv | pandas: {tiempo_csv:.4f} segundos")


In [ ]:
#Prueba de carga de los 2 ficheros (.parquet + .csv) con polars
import polars as pl    #>pip install polars
import time
import os

#0. Path de los ficheros
parquet_file = r"C:\PCAD\yellow_tripdata_2025-01.parquet"
csv_file = r"C:\PCAD\taxis.csv"

#1. Tiempo de carga del parquet con polars
start_parquet = time.time()
df_parquet = pl.read_parquet(parquet_file)
end_parquet = time.time()
tiempo_parquet = end_parquet - start_parquet

#2. Tiempo de carga de csv con polars
start_csv = time.time()
df_csv = pl.read_csv(csv_file)
end_csv = time.time()
tiempo_csv = end_csv - start_csv

#3. Mostrar los resultados
print(f"Tiempo de carga con parquet | polars: {tiempo_parquet:.4f} segundos")
print(f"Tiempo de carga con csv | polars: {tiempo_csv:.4f} segundos")

In [ ]:
#Ver el numero de registros
len(df_parquet)

In [ ]:
#Ver los primeros 50 registros
df_parquet.head(50)

In [ ]:
#Ver los campos de un fichero sin cargarlo, con polars
import polars as pl

#0. Path de los ficheros
parquet_file = r"C:\PCAD\yellow_tripdata_2025-01.parquet"

#1. Cargar el schema del conjunto de datos
esquema = pl.read_parquet_schema(parquet_file)

#2. Ver el resultado
#print(esquema)
esquema

Con polars podremos trabajar en 2 modos, modo eager (inmediato) o en modo Lazy (Plan de ejecución)
- Modo eager (paso a paso):
- carga el fichero
- filtra
- crea un dataframe temporal
- agrupa
- crea un dataframe temporal
- calcula la metrica (sum, avg, max, min, count)
- ordena
- devuelve el resultado
Cada linea ejecuta el trabajo de manera inmediata
(sin optimización global)

In [ ]:
#Vamos a calcular la propina media y el ticket medio por numero de pasajeros de los taxis en NYC, (el numero max creo que son 9), con fichero parquet, sera en modo eager
import polars as pl

#0. Path de los ficheros
parquet_file = r"C:\PCAD\yellow_tripdata_2025-01.parquet"

#1. Carga inmediata en modo eager
df_taxi_01 = pl.read_parquet(parquet_file)

#2. Comprobar si veo los primeros 20 registros
df_taxi_01.head(20)

#3. Vamos a filtrar + agrupar + agregar + ordenar
resultado = (
    df_taxi_01
    .filter(
        (pl.col("trip_distance")>1) &
        (pl.col("total_amount")>0) &
        (pl.col("passenger_count").is_not_null())
    )
    .group_by("passenger_count")
    .agg(
        pl.len().alias("Numero viajes"),   #Numero de viajes
        pl.col("trip_distance").mean().round(2).alias("distancia media"),                           #Distancia media
        pl.col("total_amount").mean().round(2).alias("ticket medio"),                            #El precio medio viaje
        pl.col("tip_amount").mean().round(2).alias("Propina media")                             #propina media`
    )
    .sort("passenger_count",descending=True)
)
resultado

- Polars permite concat igual que pandas
- Nombres de campos a usar:
  passenger_count = numero de pasajeros
  trip_distance   = Distancia del trayecto
  fare_amount     = Tarifa base
  total_amount    = Total pagado
  DOLocationID    = Zona de Destino

Practica 4 (Hihg level):
Necesitamos averiguar el numero de viajes, el precio medio de total_amount, el precio medio de fare_amount por Zona de destino
siempre y cuando el numero de pasajeros sea mayor de 2, la distancia del trayecto mayor de 2.5 miles, fare_amount sea mayor de 20, de los meses de enero a marzo del 2025. Y medir el tiempo total de proceso, desde la carga hasta el resultado. 

In [ ]:
#Solución de la practica 4 (Modo eager)
import polars as pl
import time

#0. Path de los ficheros
viajes_enero = r"C:\PCAD\yellow_tripdata_2025-01.parquet"
viajes_febrero = r"C:\PCAD\yellow_tripdata_2025-02.parquet"
viajes_marzo = r"C:\PCAD\yellow_tripdata_2025-03.parquet"

#1. Ajustar la variable de inicio_carga
inicio_carga = time.time()

#2. Cargar los 3 dataframe de polars por individual
df_taxi_01 = pl.read_parquet(viajes_enero)
df_taxi_02 = pl.read_parquet(viajes_febrero)
df_taxi_03 = pl.read_parquet(viajes_marzo)

#3. Comprobar la carga de los df's
#print(df_taxi_01.head(3))
#print(df_taxi_02.head(3))
#print(df_taxi_03.head(3))

#4. Concatenar los 3 dataframes para tener todos los datos en uno solo
df_taxi_1T = pl.concat([df_taxi_01,df_taxi_02,df_taxi_03])

#5. Ver el numero de registros
len(df_taxi_1T)

#6. Obtener el resultado, despues de filtrar + agrupar + agregar
resultado = (
    df_taxi_1T
    .filter(
        (pl.col("passenger_count")>2) &
        (pl.col("trip_distance")>2.5) &
        (pl.col("fare_amount")>20)
    )
    .group_by("DOLocationID") 
    .agg(
        pl.len().alias("Numero de viajes"),
        pl.col("total_amount").mean().round(2).alias("Promedio del total"),
        pl.col("fare_amount").mean().round(2).alias("Promedio tarifa base")     
    )
    .sort("Promedio del total")
)
#el numero de viajes, el precio medio de total_amount, el precio medio de fare_amount por Zona de destino

#7. Visualizar los datos
print(resultado)

#8. Ajustar el current time
fin_carga = time.time()

#9. Calcular el tiempo del proceso + mostrarlo
tiempo_proceso = fin_carga - inicio_carga
print(f"El tiempo total del proceso es {tiempo_proceso:.4f} segundos")

In [ ]:
#Solución practica 4 (Modo Lazy, con optimización)
#Vamos a partir del parquet de enero 20025
#En modo lazy no cargamos o ejecutamos acciones hasta la instrucción collect
#por tanto, no haremos read, si no scan

#0. Path de los ficheros
viajes_enero = r"C:\PCAD\yellow_tripdata_2025-01.parquet"

#1. Scanear el fichero de parquet
consulta = (
    pl.scan_parquet(viajes_enero)
    .filter(
        (pl.col("passenger_count")>2) &
        (pl.col("trip_distance")>2.5) &
        (pl.col("fare_amount")>20)
    )
    .group_by("DOLocationID") 
    .agg(
        pl.len().alias("Numero de viajes"),
        pl.col("total_amount").mean().round(2).alias("Promedio del total"),
        pl.col("fare_amount").mean().round(2).alias("Promedio tarifa base")     
    )
    .sort("Promedio del total")
)

#2. Mostrar el resultado del plan
consulta.explain()

#3. Como ejeuctar
resultado_lazy = consulta.collect()

#4. Ver o mostrar resultado
resultado_lazy

In [ ]:
#Solución practica 4 (Modo Lazy, con los 3 meses)
#Vamos a partir del parquet de enero 20025
#En modo lazy no cargamos o ejecutamos acciones hasta la instrucción collect
#por tanto, no haremos read, si no scan
import os
import time

#0. Ruta de la carpeta
carpeta_parquet = r"C:\PCAD"

#1. Ajustar la variable de inicio_carga
inicio_carga = time.time()

#2. Scanear el fichero de parquet
consulta = (
    pl.scan_parquet(os.path.join(carpeta_parquet,
                                 "yellow_tripdata_2025-0[1-3].parquet"))
    .filter(
        (pl.col("passenger_count")>2) &
        (pl.col("trip_distance")>2.5) &
        (pl.col("fare_amount")>20)
    )
    .group_by("DOLocationID") 
    .agg(
        pl.len().alias("Numero de viajes"),
        pl.col("total_amount").mean().round(2).alias("Promedio del total"),
        pl.col("fare_amount").mean().round(2).alias("Promedio tarifa base")     
    )
    .sort("Promedio del total")
)

#3. Mostrar el resultado del plan
consulta.explain()

#4. Como ejeuctar
resultado_lazy = consulta.collect()

#5. Ver o mostrar resultado
print(resultado_lazy)

#6. Ajustar el current time
fin_carga = time.time()

#7. Calcular el tiempo del proceso + mostrarlo
tiempo_proceso = fin_carga - inicio_carga
print(f"El tiempo total del proceso es {tiempo_proceso:.4f} segundos")



In [ ]:
#El objetivo de la practica es trabajar con el merge, sobre el excel de datos
#telefonia cargamos los primeros 3 meses (Enero-Marzo)
import pandas as pd
ruta_fichero = r"C:\PCAD\Datos Telefonia Separados Meses Comerciales URL.xlsx"
fras_enero = pd.read_excel(ruta_fichero,sheet_name="Facturación Enero 2020", header=2)
fras_febrero = pd.read_excel(ruta_fichero,sheet_name="Facturación Febrero 2020", header=0)
fras_marzo = pd.read_excel(ruta_fichero,sheet_name="Facturación Marzo 2020", header=1)
franjas = pd.read_excel(ruta_fichero,sheet_name="Franja_Edades", header=0)
#Concatenar los 3 meses 
fras_trimestre = pd.concat([fras_enero,fras_febrero,fras_marzo],ignore_index=True)
#Eliminar las filas en blanco
fras_trimestre = fras_trimestre.dropna(how='all')
#Vamos a calcular la edad + la decada
fras_trimestre['Dias'] = (pd.Timestamp('today') - fras_trimestre['Fecha Nacimiento']).dt.days
fras_trimestre['Edad'] = fras_trimestre['Dias'] // 365

#2. Mostrar el dataframe
fras_trimestre


,Nombre,Servicios,Genero,Fecha Nacimiento,Localidad,Fecha factura,Importe factura,Satisfacción,IdComercial,Dias,Edad
0,Cliente 1,Móvil,Masculino,1969-12-24,Madrid,2020-01-01,92.0,10.0,6.0,20585,56
1,Cliente 10,Voz IP,Femenino,1989-06-26,Baleares,2020-01-01,8.0,4.0,9.0,13461,36
2,Cliente 100,ADSL,Masculino,1989-09-23,Madrid,2020-01-01,49.0,7.0,3.0,13372,36
3,Cliente 1000,Móvil,Masculino,1950-10-16,Madrid,2020-01-01,25.0,7.0,7.0,27594,75
4,Cliente 1001,Voz IP,Masculino,1966-09-18,Castilla La Mancha,2020-01-01,40.0,4.0,3.0,21778,59
...,...,...,...,...,...,...,...,...,...,...,...
6169,Cliente 2008,Fibra,Femenino,1992-04-13,Barcelona,2020-03-01,65.0,3.0,3.0,12439,34
6170,Cliente 2009,ADSL,Masculino,1963-06-29,Barcelona,2020-03-01,71.0,7.0,5.0,22955,62
6171,Cliente 2010,Móvil + Fijo,Masculino,1976-01-17,Barcelona,2020-03-01,49.0,2.0,7.0,18370,50
6172,Cliente 2011,Fibra,Femenino,1982-07-10,Barcelona,2020-03-01,53.0,8.0,2.0,16004,43
